In [ ]:
# Imports
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# 1. Cargar el dataset desde el archivo CSV y preprocesar

df = pd.read_csv('/content/sample_data/titanic.csv')

# Preprocesamiento de los datos para que coincidan con las 7 características esperadas
# 1. 'Sex' a numérico (Male: 0, Female: 1)
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

# 2. Rellenar valores nulos en 'Age' con la mediana
df['age'] = df['age'].fillna(df['age'].median())

# 3. 'Embarked' a one-hot encoding
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0]) # Rellenar nulos antes de one-hot
df = pd.get_dummies(df, columns=['embarked'], prefix='Embarked')

# Asegurarse de que tenemos 3 columnas para Embarked (S, C, Q) incluso si una falta en el dataset
# Esto es importante para mantener consistencia si el conjunto de prueba tuviera diferentes valores
# o si al hacer train_test_split algunas columnas Embarked no aparecieran en un subconjunto.
# Las 3 columnas seran Embarked_C, Embarked_Q, Embarked_S
# Si un valor original de 'Embarked' no existe, la columna one-hot para ese valor será todo ceros.

# Las características originales son:
# [sexo, clase, edad, n_hermanos_esposos, n_padres_hijos, tarifa, embarcado(3 columnas)]
# Por lo tanto, necesitamos seleccionar 'Sex', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S'

# Features
X = df[['sex', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S']]

# Target
y = df['survived']

# Dividir el dataset en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Convertir a arrays de NumPy para Keras y asegurar dtype float32
titanic_train = X_train.to_numpy().astype(np.float32)
titanic_labels_train = y_train.to_numpy()
titanic_test = X_test.to_numpy().astype(np.float32)
titanic_labels_test = y_test.to_numpy()

In [ ]:
# Las características originales son:
# [sexo, clase, edad, n_hermanos_esposos, n_padres_hijos, tarifa, embarcado]
# Ya vienen preprocesadas numéricamente

print(f"Forma de los datos de entrenamiento: {titanic_train.shape}")
print(f"Forma de las etiquetas: {titanic_labels_train.shape}")
print(f"Ejemplo de una fila: {titanic_train[0]} -> Sobrevivió: {titanic_labels_train[0]}")

In [ ]:
# 2. Definir la arquitectura de la red neuronal
model = keras.Sequential([
    layers.Input(shape=(9,)),              # 9 características de entrada (actualizado de 7)
    layers.Dense(32, activation='relu'),   # Capa oculta 1
    layers.Dropout(0.2),                   # Regularización para evitar overfitting
    layers.Dense(16, activation='relu'),   # Capa oculta 2
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # Salida: probabilidad de sobrevivir
])

In [ ]:
# 3. Compilar el modelo
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',   # Adecuado para clasificación binaria
    metrics=['accuracy']
)
model.summary()

In [ ]:
# 4. Entrenar el modelo
history = model.fit(
    titanic_train, titanic_labels_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,        # 20% para validación
    verbose=1
)

In [ ]:
# 5. Evaluar el modelo
test_loss, test_accuracy = model.evaluate(titanic_test, titanic_labels_test, verbose=0)
print(f"\nPrecisión en el conjunto de prueba: {test_accuracy:.4f}")

In [ ]:
# 6. Visualizar el entrenamiento
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida (Loss)')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 7. Hacer predicciones
predicciones = model.predict(titanic_test[:30])

survived_preds = []
not_survived_preds = []

for i, pred in enumerate(predicciones):
    passenger_info = f"  Pasajero {i+1}: {pred[0]:.4f}"
    if pred[0] > 0.5:
        survived_preds.append(f"{passenger_info} -> Sobrevivió")
    else:
        not_survived_preds.append(f"{passenger_info} -> No sobrevivió")

print("\n--- Predicciones de Sobrevivientes ---")
if survived_preds:
    for p in survived_preds:
        print(p)
else:
    print("  Ningún pasajero predicho como sobreviviente en esta muestra.")

print("\n--- Predicciones de No Sobrevivientes ---")
if not_survived_preds:
    for p in not_survived_preds:
        print(p)
else:
    print("  Ningún pasajero predicho como no sobreviviente en esta muestra.")